# 极客项目 ch10：粒子系统

依赖：PyOpenGL + GLFW（OpenGL 3.3），无头环境不可执行，代码为完整忠实实现。对应《Python Playground》第 10 章（Particle System）火花喷泉；代码搬运自 `code/ch10_ps.py`（ps.py：ParticleSystem/Camera）与 `code/ch10_psmain.py`（psmain.py：PSMaker/Box），glutils 四函数见 ch09 内附，运行需同目录 star.png。

## ParticleSystem/Camera（ps.py）

书 10.1 原理：粒子位置由着色器按运动方程逐帧算出（`uPos + aVel·dt + ½·a·dt²`），alpha 随寿命衰减；配合公告板、深度遮罩关闭与加法混合三项渲染技术。

## 书 10.2 所需模块

PyOpenGL、numpy、glutils（原书 common，四函数内联于 ch09）；纹理用 star.png。

In [ ]:
# ===== 书 10.2 节：所需模块（ps.py） =====
import sys
import random
import math
from OpenGL.GL import *
import numpy
import glutils

## 书 10.3.4–10.3.5 GLSL 着色器

顶点着色器：由 aTime0 与 uTime 算 dt，存活粒子按运动方程（含绕喷泉轴的"扭动"旋转与公告板矩阵 bMatrix）求新位置，alpha = clamp(1 − 2·dt/uLifeTime)；片段着色器：纹理颜色 × 顶点色，用顶点色 alpha。

In [ ]:
# ===== 书 10.3.4/10.3.5 节：GLSL 着色器 strVS/strFS =====
strVS = """
#version 330 core

in vec3 aVel;
in vec3 aVert;
in float aTime0;
in vec2 aTexCoord;

uniform mat4 uMVMatrix;
uniform mat4 uPMatrix;
uniform mat4 bMatrix;
uniform float uTime;
uniform float uLifeTime;
uniform vec4 uColor;
uniform vec3 uPos;

out vec4 vCol;
out vec2 vTexCoord;

void main() {
    // set position
    float dt = uTime - aTime0;
    float alpha = clamp(1.0 - 2.0*dt/uLifeTime, 0.0, 1.0);
    if(dt < 0.0 || dt > uLifeTime || alpha < 0.01) {
        // out of sight!
        gl_Position = vec4(0.0, 0.0, -1000.0, 1.0);
    }
    else {
        // calculate new position
        vec3 accel = vec3(0.0, 0.0, -9.8);
        // apply a twist
        float PI = 3.14159265358979323846264;
        float theta = mod(100.0*length(aVel)*dt, 360.0)*PI/180.0;
        mat4 rot = mat4(
            vec4( cos(theta),  sin(theta), 0.0, 0.0),
            vec4(-sin(theta),  cos(theta), 0.0, 0.0),
            vec4( 0.0,         0.0,        1.0, 0.0),
            vec4( 0.0,         0.0,        0.0, 1.0)
        );
        // apply billboard matrix
        vec4 pos2 = bMatrix*rot*vec4(aVert, 1.0);
        // calculate position
        vec3 newPos = pos2.xyz + uPos + aVel*dt + 0.5*accel*dt*dt;
        // apply transformations
        gl_Position = uPMatrix * uMVMatrix * vec4(newPos, 1.0);
    }
    // set color
    vCol = vec4(uColor.rgb, alpha);
    // set tex coords
    vTexCoord = aTexCoord;
}
"""

strFS = """
#version 330 core

uniform sampler2D uSampler;
in vec4 vCol;
in vec2 vTexCoord;
out vec4 fragColor;

void main() {
    // get texture color
    vec4 texCol = texture(uSampler, vTexCoord);
    // multiply by set vertex color, use vertex color alpha
    fragColor = vec4(texCol.rgb*vCol.rgb, vCol.a);
}
"""

## 书 10.3.7 Camera 类

半径 10 绕喷泉竖直轴（z 轴）旋转的相机；rotate() 每步 theta +1° 重算 eye。

In [ ]:
# ===== 书 10.3.7 节：Camera 类 =====
# a simple camera class
class Camera:
    """helper class for viewing"""
    def __init__(self, eye, center, up):
        self.r = 10.0
        self.theta = 0
        self.eye = numpy.array(eye, dtype=numpy.float32)
        self.center = numpy.array(center, dtype=numpy.float32)
        self.up = numpy.array(up, dtype=numpy.float32)

    def rotate(self):
        """rotate eye by one step"""
        self.theta = (self.theta + 1) % 360
        # recalculate eye
        self.eye = self.center + numpy.array([
            self.r*math.cos(math.radians(self.theta)),
            self.r*math.sin(math.radians(self.theta)),
            0.0], dtype=numpy.float32)

## 书 10.3.1–10.3.6 ParticleSystem 类

restart() 一次建齐四组缓冲：每粒子 6 顶点的方形几何（10.3.1）、纹理坐标、时间延迟数组（10.3.2，`0.005·i` 错峰出生）、锥形分布初始速度（10.3.3，锥角 20°、速度随角度衰减）；render()（10.3.6）按相机方向算公告板矩阵，依次关深度遮罩、开加法混合，画完恢复现场。

In [ ]:
# ===== 书 10.3.1–10.3.6 节：ParticleSystem 类 =====
# particle system class
class ParticleSystem:

    def __init__(self, numP):
        self.numP = numP
        self.t = 0.0
        self.lifeTime = 5.0
        self.startPos = numpy.array([0.0, 0.0, 0.5], dtype=numpy.float32)
        self.texid = glutils.loadTexture('star.png')
        self.program = glutils.loadShaders(strVS, strFS)
        glUseProgram(self.program)

        # uniforms
        self.timeU = glGetUniformLocation(self.program, b"uTime")
        self.lifeTimeU = glGetUniformLocation(self.program, b"uLifeTime")
        self.pMatrixUniform = glGetUniformLocation(self.program, b'uPMatrix')
        self.mvMatrixUniform = glGetUniformLocation(self.program, b"uMVMatrix")
        self.bMatrixU = glGetUniformLocation(self.program, b"bMatrix")
        self.colorU = glGetUniformLocation(self.program, b"uColor")
        self.samplerU = glGetUniformLocation(self.program, b"uSampler")  # 原书在此另查了不存在的 b"uTex"（死 uniform），已删
        self.posU = glGetUniformLocation(self.program, b"uPos")

        # attributes
        self.vertIndex = glGetAttribLocation(self.program, b"aVert")
        self.texIndex = glGetAttribLocation(self.program, b"aTexCoord")
        self.time0Index = glGetAttribLocation(self.program, b"aTime0")
        self.velIndex = glGetAttribLocation(self.program, b"aVel")

        # render flags
        self.enableBillboard = True
        self.disableDepthMask = True
        self.enableBlend = True

        self.useStarTexture = True  # 预留开关，本书从未读取
        self.restart(numP)

    def step(self):
        self.t += 0.01

    # restart particle system
    def restart(self, numP):
        self.numP = numP
        self.t = 0.0
        self.lifeTime = 5.0

        # color
        self.col0 = numpy.array([random.random(), random.random(),
                                 random.random(), 1.0], dtype=numpy.float32)

        # create Vertex Arrays Object (VAO)
        self.vao = glGenVertexArrays(1)
        glBindVertexArray(self.vao)

        # vertices
        s = 0.2
        quadV = [
            -s, s, 0.0,
            -s, -s, 0.0,
            s, s, 0.0,
            s, -s, 0.0,
            s, s, 0.0,
            -s, -s, 0.0
        ]
        vertexData = numpy.array(numP*quadV, dtype=numpy.float32)
        self.vertexBuffer = glGenBuffers(1)
        glBindBuffer(GL_ARRAY_BUFFER, self.vertexBuffer)
        glBufferData(GL_ARRAY_BUFFER, 4*len(vertexData), vertexData, GL_STATIC_DRAW)

        # texture coords
        quadT = [
            0.0, 1.0,
            0.0, 0.0,
            1.0, 1.0,
            1.0, 0.0,
            1.0, 1.0,
            0.0, 0.0
        ]
        tcData = numpy.array(numP*quadT, dtype=numpy.float32)
        self.tcBuffer = glGenBuffers(1)
        glBindBuffer(GL_ARRAY_BUFFER, self.tcBuffer)
        glBufferData(GL_ARRAY_BUFFER, 4*len(tcData), tcData, GL_STATIC_DRAW)

        # time lags
        timeData = numpy.repeat(0.005*numpy.arange(numP, dtype=numpy.float32),
                                6)  # 原书写 4，应为 6：每粒子两三角形共 6 顶点
        self.timeBuffer = glGenBuffers(1)
        glBindBuffer(GL_ARRAY_BUFFER, self.timeBuffer)
        glBufferData(GL_ARRAY_BUFFER, 4*len(timeData), timeData, GL_STATIC_DRAW)

        # velocities
        velocities = []
        coneAngle = math.radians(20.0)  # 锥角 20 度
        for i in range(numP):
            # inclination
            angleRatio = random.random()
            a = angleRatio*coneAngle
            # azimuth
            t = random.random()*(2.0*math.pi)
            # get velocity on sphere
            vx = math.sin(a)*math.cos(t)
            vy = math.sin(a)*math.sin(t)
            vz = math.cos(a)
            # speed decreases with angle
            speed = 15.0*(1.0 - angleRatio*angleRatio)
            velocities += 6*[speed*vx, speed*vy, speed*vz]  # 每粒子 6 顶点共用一个速度
        self.velBuffer = glGenBuffers(1)
        glBindBuffer(GL_ARRAY_BUFFER, self.velBuffer)
        velData = numpy.array(velocities, dtype=numpy.float32)
        glBufferData(GL_ARRAY_BUFFER, 4*len(velData), velData, GL_STATIC_DRAW)

        # enable arrays
        glEnableVertexAttribArray(self.vertIndex)
        glEnableVertexAttribArray(self.texIndex)
        glEnableVertexAttribArray(self.time0Index)
        glEnableVertexAttribArray(self.velIndex)

        # set buffers
        glBindBuffer(GL_ARRAY_BUFFER, self.vertexBuffer)
        glVertexAttribPointer(self.vertIndex, 3, GL_FLOAT, GL_FALSE, 0, None)

        glBindBuffer(GL_ARRAY_BUFFER, self.tcBuffer)
        glVertexAttribPointer(self.texIndex, 2, GL_FLOAT, GL_FALSE, 0, None)

        glBindBuffer(GL_ARRAY_BUFFER, self.velBuffer)
        glVertexAttribPointer(self.velIndex, 3, GL_FLOAT, GL_FALSE, 0, None)

        glBindBuffer(GL_ARRAY_BUFFER, self.timeBuffer)
        glVertexAttribPointer(self.time0Index, 1, GL_FLOAT, GL_FALSE, 0, None)

        glBindVertexArray(0)

    # render the particle system
    def render(self, pMatrix, mvMatrix, camera):
        glUseProgram(self.program)

        glUniformMatrix4fv(self.pMatrixUniform, 1, GL_FALSE, pMatrix)
        glUniformMatrix4fv(self.mvMatrixUniform, 1, GL_FALSE, mvMatrix)

        # 公告板矩阵：让四边形面向观看方向
        if self.enableBillboard:
            N = camera.eye - camera.center  # 从场景指向眼睛 = 观看方向的反向
            N /= numpy.linalg.norm(N)
            U = camera.up
            U /= numpy.linalg.norm(U)
            R = numpy.cross(U, N)
            U2 = numpy.cross(N, R)
            # 按行铺平，传 GL_TRUE 让驱动转置成列优先
            bMatrix = numpy.array([R[0], U2[0], N[0], 0.0,
                                   R[1], U2[1], N[1], 0.0,
                                   R[2], U2[2], N[2], 0.0,
                                   0.0, 0.0, 0.0, 1.0], dtype=numpy.float32)
            glUniformMatrix4fv(self.bMatrixU, 1, GL_TRUE, bMatrix)
        else:
            bMatrix = numpy.array([1.0, 0.0, 0.0, 0.0,
                                   0.0, 1.0, 0.0, 0.0,
                                   0.0, 0.0, 1.0, 0.0,
                                   0.0, 0.0, 0.0, 1.0], dtype=numpy.float32)
            glUniformMatrix4fv(self.bMatrixU, 1, GL_FALSE, bMatrix)

        # set start position
        glUniform3fv(self.posU, 1, self.startPos)
        glUniform1f(self.timeU, self.t)
        glUniform1f(self.lifeTimeU, self.lifeTime)
        glUniform4fv(self.colorU, 1, self.col0)

        # enable texture
        glActiveTexture(GL_TEXTURE0)
        glBindTexture(GL_TEXTURE_2D, self.texid)
        glUniform1i(self.samplerU, 0)

        # turn depth mask off（黑像素不写深度，避免遮挡别的火花）
        if self.disableDepthMask:
            glDepthMask(GL_FALSE)

        # enable blending（加法混合：黑背景"消失"）
        if self.enableBlend:
            glBlendFunc(GL_SRC_ALPHA, GL_ONE)
            glEnable(GL_BLEND)

        glBindVertexArray(self.vao)
        glDrawArrays(GL_TRIANGLES, 0, 6*self.numP)
        glBindVertexArray(0)

        # restore state
        if self.enableBlend:
            glDisable(GL_BLEND)
        if self.disableDepthMask:
            glDepthMask(GL_TRUE)
        glBindTexture(GL_TEXTURE_2D, 0)

## 主程序 PSMaker + Box（psmain.py）

psmain.py 导入 ps.py 的 ParticleSystem/Camera，创建 GLFW 窗口与主循环；Box（书 10.5）是无光照暗红立方体，标出喷泉源头。键盘 R/B/D/T 逐个开关相机旋转、公告板、深度遮罩、透明混合——对照实验台（书 10.6.2）。

In [ ]:
# ===== 书 10.6 节：PSMaker——GLFW 窗口与主程序循环（10.6.1 step / 10.6.2 键盘 / 10.6.3 run） =====
import sys
import os
import math
from OpenGL.GL import *
import numpy
from ch10_ps import ParticleSystem, Camera  # 原书为 from ps import ...
import glutils
import glfw


class PSMaker:
    """GLFW Rendering window class for Particle System"""
    def __init__(self):
        self.camera = Camera([15.0, 0.0, 2.5],   # 眼睛：x=15、与喷泉同高
                             [0.0, 0.0, 2.5],    # 注视点：喷泉竖直轴上
                             [0.0, 0.0, 1.0])    # 向上向量：+z
        self.aspect = 1.0
        self.numP = 300
        self.t = 0  # 主程序时间（毫秒），只用于计时重启
        self.rotate = True

        cwd = os.getcwd()  # glfwInit 会改变当前工作目录，先保存
        glfw.glfwInit()
        os.chdir(cwd)

        glfw.glfwWindowHint(glfw.GLFW_CONTEXT_VERSION_MAJOR, 3)
        glfw.glfwWindowHint(glfw.GLFW_CONTEXT_VERSION_MINOR, 3)
        glfw.glfwWindowHint(glfw.GLFW_OPENGL_FORWARD_COMPAT, GL_TRUE)
        glfw.glfwWindowHint(glfw.GLFW_OPENGL_PROFILE, glfw.GLFW_OPENGL_CORE_PROFILE)

        self.width, self.height = 640, 480
        self.aspect = self.width/float(self.height)
        self.win = glfw.glfwCreateWindow(self.width, self.height, b"Particle System")
        glfw.glfwMakeContextCurrent(self.win)

        glViewport(0, 0, self.width, self.height)
        glEnable(GL_DEPTH_TEST)
        glClearColor(0.2, 0.2, 0.2, 1.0)  # 深灰背景：加法混合的"底色"

        glfw.glfwSetMouseButtonCallback(self.win, self.onMouseButton)
        glfw.glfwSetKeyCallback(self.win, self.onKeyboard)
        glfw.glfwSetWindowSizeCallback(self.win, self.onSize)

        self.psys = ParticleSystem(self.numP)
        self.box = Box(1.0)
        self.exitNow = False

    def onMouseButton(self, win, button, action, mods):
        pass

    def onKeyboard(self, win, key, scancode, action, mods):
        if action == glfw.GLFW_PRESS:
            if key == glfw.GLFW_KEY_ESCAPE:  # ESC 退出
                self.exitNow = True
            elif key == glfw.GLFW_KEY_R:     # R：相机旋转开关
                self.rotate = not self.rotate
            elif key == glfw.GLFW_KEY_B:     # B：公告板开关
                self.psys.enableBillboard = not self.psys.enableBillboard
            elif key == glfw.GLFW_KEY_D:     # D：深度遮罩开关
                self.psys.disableDepthMask = not self.psys.disableDepthMask
            elif key == glfw.GLFW_KEY_T:     # T：透明混合开关
                self.psys.enableBlend = not self.psys.enableBlend

    def onSize(self, win, width, height):
        self.width, self.height = width, height
        self.aspect = width/float(height)
        glViewport(0, 0, self.width, self.height)

    def step(self):
        self.t += 10
        self.psys.step()
        if self.rotate:
            self.camera.rotate()
        if not int(self.t) % 5000:  # 每 5 秒重启（换随机颜色）
            self.psys.restart(self.numP)

    def run(self):
        glfw.glfwSetTime(0)
        t = 0.0
        while not glfw.glfwWindowShouldClose(self.win) and not self.exitNow:
            currT = glfw.glfwGetTime()
            if currT - t > 0.01:  # 每 0.01 秒渲染一帧
                t = currT
                glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)
                pMatrix = glutils.perspective(100.0, self.aspect, 0.1, 100.0)
                mvMatrix = glutils.lookAt(self.camera.eye, self.camera.center,
                                          self.camera.up)
                self.box.render(pMatrix, mvMatrix)  # 不透明物体先画
                self.psys.render(pMatrix, mvMatrix, self.camera)  # 半透明最后画
                self.step()
                glfw.glfwSwapBuffers(self.win)
            glfw.glfwPollEvents()
        glfw.glfwTerminate()

In [ ]:
# ===== 书 10.5 节：Box——标出喷泉源头的暗红盒子 =====
# ===== ch10 盒子 =====（原书 box.py：无光照暗红立方体，标出喷泉源头）
strVS = """
#version 330 core
in vec3 aVert;
uniform mat4 uMVMatrix;
uniform mat4 uPMatrix;
out vec4 vCol;
void main() {
    gl_Position = uPMatrix * uMVMatrix * vec4(aVert, 1.0);
    vCol = vec4(0.8, 0.0, 0.0, 1.0);
}
"""

strFS = """
#version 330 core
in vec4 vCol;
out vec4 fragColor;
void main() {
    fragColor = vCol;
}
"""


class Box:
    def __init__(self, side):
        self.side = side

        self.program = glutils.loadShaders(strVS, strFS)
        glUseProgram(self.program)

        s = side/2.0
        vertices = [
            -s, s, -s,
            -s, -s, -s,
            s, s, -s,
            s, -s, -s,
            s, s, -s,
            -s, -s, -s,

            -s, s, s,
            -s, -s, s,
            s, s, s,
            s, -s, s,
            s, s, s,
            -s, -s, s,

            -s, -s, s,
            -s, -s, -s,
            s, -s, s,
            s, -s, -s,
            s, -s, s,
            -s, -s, -s,

            -s, s, s,
            -s, s, -s,
            s, s, s,
            s, s, -s,
            s, s, s,
            -s, s, -s,

            -s, -s, s,
            -s, -s, -s,
            -s, s, s,
            -s, s, -s,
            -s, s, s,
            -s, -s, -s,

            s, -s, s,
            s, -s, -s,
            s, s, s,
            s, s, -s,
            s, s, s,
            s, -s, -s
        ]

        self.vao = glGenVertexArrays(1)
        glBindVertexArray(self.vao)
        vertexData = numpy.array(vertices, dtype=numpy.float32)
        self.vertexBuffer = glGenBuffers(1)
        glBindBuffer(GL_ARRAY_BUFFER, self.vertexBuffer)
        glBufferData(GL_ARRAY_BUFFER, 4*len(vertexData), vertexData, GL_STATIC_DRAW)
        self.vertIndex = glGetAttribLocation(self.program, b"aVert")
        glEnableVertexAttribArray(self.vertIndex)
        glBindBuffer(GL_ARRAY_BUFFER, self.vertexBuffer)
        glVertexAttribPointer(self.vertIndex, 3, GL_FLOAT, GL_FALSE, 0, None)
        glBindVertexArray(0)

    def render(self, pMatrix, mvMatrix):
        glUseProgram(self.program)
        glUniformMatrix4fv(glGetUniformLocation(self.program, b'uPMatrix'),
                           1, GL_FALSE, pMatrix)
        glUniformMatrix4fv(glGetUniformLocation(self.program, b'uMVMatrix'),
                           1, GL_FALSE, mvMatrix)
        glBindVertexArray(self.vao)
        glDrawArrays(GL_TRIANGLES, 0, 36)
        glBindVertexArray(0)

In [ ]:
# ===== 书 10.7/10.8 节：main() 运行入口 =====
def main():
    print('starting particle system...')
    prog = PSMaker()
    prog.run()


if __name__ == '__main__':
    main()